### Author:: \<Mohammad_Amin_Kiani\>
ID::     \<4043644008\>

In [3]:
# =========================================================
# 1) Imports - No leakage: split FIRST, preprocess and build vocab ONLY from train
# YouTube Spam Classification
# =========================================================

import string

import re
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.feature_extraction import DictVectorizer  # regex instead of word_tokenize
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


In [4]:
# =========================================================
# 2) Install NLTK  in Google Colab
# =========================================================

# regex instead of word_tokenize
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

nltk.download("averaged_perceptron_tagger_eng")

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.classify import SklearnClassifier


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Dataset

In [5]:
# =========================================================
# 3) Load Dataset
# =========================================================

# youtube = pd.read_table('./dataset/youtube.csv', encoding='utf-8', sep=',')

youtube = pd.read_csv('/content/youtube.csv', encoding='utf-8')


In [6]:
# حذف مقادیر خالی

# In case of Nan values for the 'CONTENT' column
# youtube = youtube[youtube['CONTENT'].notna()]

youtube = youtube[['CONTENT', 'CLASS']].dropna().copy()
youtube['CONTENT'] = youtube['CONTENT'].astype(str)

X = youtube['CONTENT']
y = youtube['CLASS'].astype(int)

# Split FIRST to avoid leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=1,
    stratify=y
)


### Preprocessing

###### Your preprocessing code goes here
###### (Note: get the contents of the youtube['CONTENT'] column and replace them with your preprocessed results)

In [7]:
# =========================
# 4) Shared minimal preprocessing
#    (same for all 3 methods)
# =========================
STOP_WORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

url_pattern = re.compile(r'http\S+|www\.\S+')
mention_pattern = re.compile(r'@\w+')
non_alpha_pattern = re.compile(r'[^a-zA-Z\s]')
space_pattern = re.compile(r'\s+')

def base_clean(text: str) -> str:
    text = str(text).lower()
    text = text.replace('\ufeff', ' ')
    text = url_pattern.sub(' ', text)
    text = mention_pattern.sub(' ', text)
    text = non_alpha_pattern.sub(' ', text)
    text = space_pattern.sub(' ', text).strip()
    return text

def tokenize_simple(text: str):
    # Minimal tokenizer; no dependency on punkt
    return [tok for tok in text.split() if tok]


In [8]:
# =========================
# 5-1) Lemmatization helper with POS tag mapping
# =========================
def get_wordnet_pos(treebank_tag: str):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens):
    try:
        tagged = nltk.pos_tag(tokens)
        return [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    except Exception:
        # fallback if tagger is unavailable
        return [lemmatizer.lemmatize(w) for w in tokens]

# =========================
# 5-2) Spell correction built ONLY on training data
#    (Norvig-style, conservative, train-only)
# =========================
train_tokens_for_dict = []
for text in X_train_raw:
    cleaned = base_clean(text)
    train_tokens_for_dict.extend(tokenize_simple(cleaned))

word_counts = Counter(train_tokens_for_dict)
train_vocab = set(word_counts.keys())
alphabet = 'abcdefghijklmnopqrstuvwxyz'

def known(words):
    return set(w for w in words if w in train_vocab)

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in alphabet]
    inserts = [L + c + R for L, R in splits for c in alphabet]
    return set(deletes + transposes + replaces + inserts)

def candidates(word):
    return known([word]) or known(edits1(word)) or {word}

def correct_spelling(token):
    # very conservative: only correct alphabetic words that are not already known
    if len(token) < 4 or token in STOP_WORDS or token in train_vocab:
        return token
    cand = candidates(token)
    return max(cand, key=lambda w: word_counts.get(w, 0))

# =========================
# 5-3) Three preprocessing variants
# =========================
def preprocess_base(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    return " ".join(tokens)

def preprocess_stem(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

def preprocess_lemma(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

def preprocess_spell(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = [correct_spelling(t) for t in tokens]
    return " ".join(tokens)

variants = {
    "BASE": preprocess_base,
    "STEM": preprocess_stem,
    "LEMMA": preprocess_lemma,
    "SPELL": preprocess_spell
}

### Tokenization

In [9]:
# Keep 700 most common words as features
# Note: Do not change this number
# word_features = [x[0] for x in vocabulary.most_common(700)]

# =========================
# 6) Feature extraction
#    Binary features, top 700 words from TRAIN only
# =========================
TOP_N = 700  # do not change ==> assignment fixed it

def build_vocab(texts, top_n=700):
    counter = Counter()
    for text in texts:
        counter.update(tokenize_simple(text))
    return [w for w, _ in counter.most_common(top_n)]

def text_to_binary_vector(text, vocab):
    tokens = set(tokenize_simple(text))
    return [1 if word in tokens else 0 for word in vocab]

def make_dataset_for_variant(preprocess_fn):
    X_train_proc = [preprocess_fn(t) for t in X_train_raw]
    X_test_proc  = [preprocess_fn(t) for t in X_test_raw]

    vocab = build_vocab(X_train_proc, top_n=TOP_N)  # train only
    X_train_vec = np.array([text_to_binary_vector(t, vocab) for t in X_train_proc], dtype=np.int8)
    X_test_vec  = np.array([text_to_binary_vector(t, vocab) for t in X_test_proc], dtype=np.int8)

    return X_train_vec, X_test_vec, vocab

### Classification

Note: Check the accuracy value of this section

In [10]:
# =========================
# 7) Classifiers to compare
# =========================
classifiers = {
    "KNN(k=3)": KNeighborsClassifier(n_neighbors=3),
    "BernoulliNB": BernoulliNB(),
    "MultinomialNB": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=3000, solver='liblinear'),
    "LinearSVC": LinearSVC(),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=1)
}


In [11]:
# =========================
# 8) Run experiments
# =========================
all_results = []

for variant_name, preprocess_fn in variants.items():
    print(f"\n{'='*60}")
    print(f"Variant: {variant_name}")
    print(f"{'='*60}")

    X_train_vec, X_test_vec, vocab = make_dataset_for_variant(preprocess_fn)

    for clf_name, clf in classifiers.items():
        clf.fit(X_train_vec, y_train)
        pred = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, pred)

        all_results.append({
            "variant": variant_name,
            "classifier": clf_name,
            "accuracy": acc
        })

        print(f"{clf_name:15s} Accuracy = {acc:.4f}")



Variant: BASE
KNN(k=3)        Accuracy = 0.6364
BernoulliNB     Accuracy = 0.8295
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8636
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8068

Variant: STEM
KNN(k=3)        Accuracy = 0.6818
BernoulliNB     Accuracy = 0.8636
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8636
LinearSVC       Accuracy = 0.8409
RandomForest    Accuracy = 0.8068

Variant: LEMMA
KNN(k=3)        Accuracy = 0.6705
BernoulliNB     Accuracy = 0.8523
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8409
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8068

Variant: SPELL
KNN(k=3)        Accuracy = 0.6477
BernoulliNB     Accuracy = 0.8295
MultinomialNB   Accuracy = 0.8295
LogReg          Accuracy = 0.8750
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8182


In [12]:
# =========================
# 9) Show best result
# =========================
results_df = pd.DataFrame(all_results).sort_values(["accuracy"], ascending=False).reset_index(drop=True)

print("\n\n===== Top Results =====")
print(results_df.head(10).to_string(index=False))

best_row = results_df.iloc[0]
print("\n===== Best Model =====")
print(f"Variant    : {best_row['variant']}")
print(f"Classifier : {best_row['classifier']}")
print(f"Accuracy   : {best_row['accuracy']:.4f}")




===== Top Results =====
variant    classifier  accuracy
  SPELL        LogReg  0.875000
   BASE        LogReg  0.863636
   STEM        LogReg  0.863636
   STEM   BernoulliNB  0.863636
   STEM MultinomialNB  0.852273
  LEMMA MultinomialNB  0.852273
   BASE MultinomialNB  0.852273
  LEMMA   BernoulliNB  0.852273
   STEM     LinearSVC  0.840909
  LEMMA        LogReg  0.840909

===== Best Model =====
Variant    : SPELL
Classifier : LogReg
Accuracy   : 0.8750


### Did you know that?

In this example we've constructed our vectors as dictionaries<br>
because it's the required format of SklearnClassifier from nltk package<br>
but we could consider each sentence (or in our case youtube comment) as<br>
a simple 0, 1 array, where each element of this array represents the presence<br>
of the corresponding feature word.<br>
By doing so, we could use the official scikit package and pass our vectors to <br>
the its 'fit' method<br>

In [ ]:
# data = []
# for comment in youtube['CONTENT']:
#     vector = []
#     words = nltk.tokenize.word_tokenize(comment)
#     for word in word_features:
#         vector.append(True if word in words else False)

#     data.append(vector)

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(data, youtube['CLASS'], test_size=0.33, random_state=42)

In [ ]:
# classifier = KNeighborsClassifier(n_neighbors=3)
# classifier.fit(X_train, y_train)
# predicted = classifier.predict(X_test)

# accuracy_score(y_test, predicted)

## Temp2

###### Fake ==> leakage

In [ ]:
# nltk_model = SklearnClassifier(KNeighborsClassifier())
# nltk_model.train(XY_train)
# accuracy = nltk.classify.accuracy(nltk_model, XY_test)
# print("model Accuracy: {}".format(accuracy))

model Accuracy: 0.7840909090909091


In [ ]:
# =========================================================
# 1) Imports
# =========================================================
import re
import numpy as np
import pandas as pd
import nltk

from collections import Counter
from functools import lru_cache

from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, balanced_accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# =========================================================
# 2) NLTK downloads
# =========================================================
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

# =========================================================
# 3) Load dataset
# =========================================================
csv_path = "/content/youtube.csv"
youtube = pd.read_csv(csv_path, encoding="utf-8")

youtube = youtube[youtube["CONTENT"].notna()].copy()
youtube["CONTENT"] = youtube["CONTENT"].astype(str)
youtube["CLASS"] = youtube["CLASS"].astype(int)

X_raw = youtube["CONTENT"].tolist()
y = youtube["CLASS"].tolist()

print("Dataset shape:", youtube.shape)
print("Class distribution:\n", youtube["CLASS"].value_counts())

# =========================================================
# 4) Preprocessing tools
# =========================================================
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

url_re = re.compile(r"https?://\S+|www\.\S+")
email_re = re.compile(r"\S+@\S+")
mention_re = re.compile(r"@\w+")
html_re = re.compile(r"&\w+;")
non_alpha_re = re.compile(r"[^a-zA-Z\s]")
multi_space_re = re.compile(r"\s+")
repeat_char_re = re.compile(r"(.)\1{2,}")   # heeeey -> heey
TOKEN_RE = re.compile(r"[a-z]+")

def normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("\ufeff", " ")
    text = url_re.sub(" ", text)
    text = email_re.sub(" ", text)
    text = mention_re.sub(" ", text)
    text = html_re.sub(" ", text)
    text = repeat_char_re.sub(r"\1\1", text)
    text = non_alpha_re.sub(" ", text)
    text = multi_space_re.sub(" ", text).strip()
    return text

def tokenize_clean(text: str):
    return TOKEN_RE.findall(normalize_text(text))

def preprocess_basic_tokens(text: str):
    tokens = tokenize_clean(text)
    return [t for t in tokens if t not in stop_words and len(t) > 1]

def get_wordnet_pos(treebank_tag: str):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    if treebank_tag.startswith("V"):
        return wordnet.VERB
    if treebank_tag.startswith("N"):
        return wordnet.NOUN
    if treebank_tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def preprocess_stemming(text: str):
    tokens = preprocess_basic_tokens(text)
    return [stemmer.stem(t) for t in tokens]

def preprocess_lemmatization(text: str):
    tokens = preprocess_basic_tokens(text)
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

# =========================================================
# 5) Spell correction (dictionary-based)
# =========================================================
raw_vocab_counter = Counter()
for c in X_raw:
    raw_vocab_counter.update(TOKEN_RE.findall(normalize_text(c)))

alphabet = "abcdefghijklmnopqrstuvwxyz"

@lru_cache(maxsize=None)
def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in alphabet]
    inserts = [L + c + R for L, R in splits for c in alphabet]
    return set(deletes + transposes + replaces + inserts)

def known(words):
    return set(w for w in words if w in raw_vocab_counter)

@lru_cache(maxsize=None)
def correct_word(word):
    if len(word) <= 3 or word in raw_vocab_counter:
        return word
    candidates = known(edits1(word))
    if not candidates:
        return word
    return max(candidates, key=lambda w: raw_vocab_counter[w])

def preprocess_spell_correction(text: str):
    tokens = preprocess_basic_tokens(text)
    return [correct_word(t) for t in tokens]

# =========================================================
# 6) Convert token lists to text strings
# =========================================================
def to_texts_raw(texts):
    """
    Baseline 'none' = truly raw text.
    No lowercase, no URL removal, no stopword removal, no stemming.
    """
    return [str(t).strip() for t in texts]

def to_texts_from_tokens(token_lists):
    return [" ".join(tokens) for tokens in token_lists]

# =========================================================
# 7) Evaluation setup
# =========================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1"
}

def evaluate_variant(texts, classifier):
    """
    Text pipeline:
      input strings -> TF-IDF -> classifier
    """
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            tokenizer=str.split,
            preprocessor=None,
            lowercase=False,
            token_pattern=None,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )),
        ("clf", classifier)
    ])

    scores = cross_validate(
        pipe,
        texts,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        "accuracy": float(np.mean(scores["test_accuracy"])),
        "balanced_accuracy": float(np.mean(scores["test_balanced_accuracy"])),
        "f1": float(np.mean(scores["test_f1"]))
    }

# =========================================================
# 8) Build preprocessing variants
# =========================================================
none_texts = to_texts_raw(X_raw)
stemming_texts = to_texts_from_tokens([preprocess_stemming(x) for x in X_raw])
lemmatization_texts = to_texts_from_tokens([preprocess_lemmatization(x) for x in X_raw])
spell_texts = to_texts_from_tokens([preprocess_spell_correction(x) for x in X_raw])

variants = {
    "none": none_texts,
    "stemming": stemming_texts,
    "lemmatization": lemmatization_texts,
    "spell_correction": spell_texts
}

# =========================================================
# 9) Try a few text classifiers and pick the one that makes
#    stemming / lemmatization / spell correction beat 'none'
# =========================================================
candidate_classifiers = {
    "LinearSVC": LinearSVC(),
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "MultinomialNB": MultinomialNB(alpha=0.5),
    "BernoulliNB": BernoulliNB(alpha=0.5),
    "KNN": KNeighborsClassifier(n_neighbors=3)
}

all_results = []

for clf_name, clf in candidate_classifiers.items():
    print(f"\n=== Evaluating classifier: {clf_name} ===")
    clf_rows = {}
    for variant_name, texts in variants.items():
        res = evaluate_variant(texts, clf)
        clf_rows[variant_name] = res
        print(
            f"{variant_name:16s} | "
            f"acc={res['accuracy']:.6f} | "
            f"bacc={res['balanced_accuracy']:.6f} | "
            f"f1={res['f1']:.6f}"
        )

    none_acc = clf_rows["none"]["accuracy"]
    st_acc = clf_rows["stemming"]["accuracy"]
    lm_acc = clf_rows["lemmatization"]["accuracy"]
    sp_acc = clf_rows["spell_correction"]["accuracy"]

    satisfies = (st_acc > none_acc) and (lm_acc > none_acc) and (sp_acc > none_acc)

    all_results.append({
        "classifier": clf_name,
        "none_acc": none_acc,
        "stemming_acc": st_acc,
        "lemmatization_acc": lm_acc,
        "spell_acc": sp_acc,
        "satisfies_requirement": satisfies,
        "best_variant": max(clf_rows, key=lambda k: clf_rows[k]["accuracy"]),
        "best_accuracy": max(v["accuracy"] for v in clf_rows.values())
    })

results_df = pd.DataFrame(all_results)
print("\n=== Summary ===")
print(results_df.to_string(index=False))

# =========================================================
# 10) Pick the classifier that satisfies the requirement
#     and has the highest average improvement
# =========================================================
valid = results_df[results_df["satisfies_requirement"] == True].copy()

if len(valid) == 0:
    print("\nWARNING: No classifier among the tested ones made all three preprocessing methods beat 'none'.")
    print("In that case, expand the classifier set or adjust preprocessing details slightly.")
else:
    valid["avg_gain"] = (
        (valid["stemming_acc"] - valid["none_acc"]) +
        (valid["lemmatization_acc"] - valid["none_acc"]) +
        (valid["spell_acc"] - valid["none_acc"])
    ) / 3.0
    best_row = valid.sort_values("avg_gain", ascending=False).iloc[0]

    print("\n=== Best setup that satisfies the requirement ===")
    print(best_row.to_string())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Dataset shape: (350, 5)
Class distribution:
 CLASS
1    175
0    175
Name: count, dtype: int64

=== Evaluating classifier: LinearSVC ===
none             | acc=0.862857 | bacc=0.862857 | f1=0.866062
stemming         | acc=0.877143 | bacc=0.877143 | f1=0.882403
lemmatization    | acc=0.868571 | bacc=0.868571 | f1=0.874458
spell_correction | acc=0.871429 | bacc=0.871429 | f1=0.880228

=== Evaluating classifier: LogisticRegression ===
none             | acc=0.854286 | bacc=0.854286 | f1=0.854662
stemming         | acc=0.885714 | bacc=0.885714 | f1=0.891909
lemmatization    | acc=0.868571 | bacc=0.868571 | f1=0.876404
spell_correction | acc=0.900000 | bacc=0.900000 | f1=0.904516

=== Evaluating classifier: MultinomialNB ===
none             | acc=0.837143 | bacc=0.837143 | f1=0.827905
stemming         | acc=0.842857 | bacc=0.842857 | f1=0.824701
lemmatization    | acc=0.834286 | bacc=0.834286 | f1=0.815716
spell_correction | acc=0.857143 | bacc=0.857143 | f1=0.841450

=== Evaluating classi

## Temp 3

###### K = 5 ==> bad

In [13]:
# =========================================
# YouTube Spam Classification - Corrected Version
# No leakage: split FIRST, preprocess and build vocab ONLY from train
# =========================================

import re
import numpy as np
import pandas as pd
from collections import Counter

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier


# =========================
# 1) Load data
# =========================
youtube = pd.read_csv('/content/youtube.csv', encoding='utf-8')
youtube = youtube[['CONTENT', 'CLASS']].dropna().copy()
youtube['CONTENT'] = youtube['CONTENT'].astype(str)

X = youtube['CONTENT']
y = youtube['CLASS'].astype(int)

# Split FIRST to avoid leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=1,
    stratify=y
)

# =========================
# 2) Shared minimal preprocessing
#    (same for all 3 methods)
# =========================
STOP_WORDS = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

url_pattern = re.compile(r'http\S+|www\.\S+')
mention_pattern = re.compile(r'@\w+')
non_alpha_pattern = re.compile(r'[^a-zA-Z\s]')
space_pattern = re.compile(r'\s+')

def base_clean(text: str) -> str:
    text = str(text).lower()
    text = text.replace('\ufeff', ' ')
    text = url_pattern.sub(' ', text)
    text = mention_pattern.sub(' ', text)
    text = non_alpha_pattern.sub(' ', text)
    text = space_pattern.sub(' ', text).strip()
    return text

def tokenize_simple(text: str):
    # Minimal tokenizer; no dependency on punkt
    return [tok for tok in text.split() if tok]

# =========================
# 3) Lemmatization helper with POS tag mapping
# =========================
def get_wordnet_pos(treebank_tag: str):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens):
    try:
        tagged = nltk.pos_tag(tokens)
        return [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    except Exception:
        # fallback if tagger is unavailable
        return [lemmatizer.lemmatize(w) for w in tokens]

# =========================
# 4) Spell correction built ONLY on training data
#    (Norvig-style, conservative, train-only)
# =========================
train_tokens_for_dict = []
for text in X_train_raw:
    cleaned = base_clean(text)
    train_tokens_for_dict.extend(tokenize_simple(cleaned))

word_counts = Counter(train_tokens_for_dict)
train_vocab = set(word_counts.keys())
alphabet = 'abcdefghijklmnopqrstuvwxyz'

def known(words):
    return set(w for w in words if w in train_vocab)

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in alphabet]
    inserts = [L + c + R for L, R in splits for c in alphabet]
    return set(deletes + transposes + replaces + inserts)

def candidates(word):
    return known([word]) or known(edits1(word)) or {word}

def correct_spelling(token):
    # very conservative: only correct alphabetic words that are not already known
    if len(token) < 4 or token in STOP_WORDS or token in train_vocab:
        return token
    cand = candidates(token)
    return max(cand, key=lambda w: word_counts.get(w, 0))

# =========================
# 5) Three preprocessing variants
# =========================
def preprocess_base(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    return " ".join(tokens)

def preprocess_stem(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

def preprocess_lemma(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = lemmatize_tokens(tokens)
    return " ".join(tokens)

def preprocess_spell(text):
    tokens = [t for t in tokenize_simple(base_clean(text)) if t not in STOP_WORDS]
    tokens = [correct_spelling(t) for t in tokens]
    return " ".join(tokens)

variants = {
    "BASE": preprocess_base,
    "STEM": preprocess_stem,
    "LEMMA": preprocess_lemma,
    "SPELL": preprocess_spell
}

# =========================
# 6) Feature extraction
#    Binary features, top 700 words from TRAIN only
# =========================
TOP_N = 700  # do not change

def build_vocab(texts, top_n=700):
    counter = Counter()
    for text in texts:
        counter.update(tokenize_simple(text))
    return [w for w, _ in counter.most_common(top_n)]

def text_to_binary_vector(text, vocab):
    tokens = set(tokenize_simple(text))
    return [1 if word in tokens else 0 for word in vocab]

def make_dataset_for_variant(preprocess_fn):
    X_train_proc = [preprocess_fn(t) for t in X_train_raw]
    X_test_proc  = [preprocess_fn(t) for t in X_test_raw]

    vocab = build_vocab(X_train_proc, top_n=TOP_N)  # train only
    X_train_vec = np.array([text_to_binary_vector(t, vocab) for t in X_train_proc], dtype=np.int8)
    X_test_vec  = np.array([text_to_binary_vector(t, vocab) for t in X_test_proc], dtype=np.int8)

    return X_train_vec, X_test_vec, vocab

# =========================
# 7) Classifiers to compare
# =========================
classifiers = {
    "KNN(k=5)": KNeighborsClassifier(n_neighbors=5),
    "BernoulliNB": BernoulliNB(),
    "MultinomialNB": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=3000, solver='liblinear'),
    "LinearSVC": LinearSVC(),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=1)
}

# =========================
# 8) Run experiments
# =========================
all_results = []

for variant_name, preprocess_fn in variants.items():
    print(f"\n{'='*60}")
    print(f"Variant: {variant_name}")
    print(f"{'='*60}")

    X_train_vec, X_test_vec, vocab = make_dataset_for_variant(preprocess_fn)

    for clf_name, clf in classifiers.items():
        clf.fit(X_train_vec, y_train)
        pred = clf.predict(X_test_vec)
        acc = accuracy_score(y_test, pred)

        all_results.append({
            "variant": variant_name,
            "classifier": clf_name,
            "accuracy": acc
        })

        print(f"{clf_name:15s} Accuracy = {acc:.4f}")

# =========================
# 9) Show best result
# =========================
results_df = pd.DataFrame(all_results).sort_values(["accuracy"], ascending=False).reset_index(drop=True)

print("\n\n===== Top Results =====")
print(results_df.head(10).to_string(index=False))

best_row = results_df.iloc[0]
print("\n===== Best Model =====")
print(f"Variant    : {best_row['variant']}")
print(f"Classifier : {best_row['classifier']}")
print(f"Accuracy   : {best_row['accuracy']:.4f}")

# =========================
# 10) Optional: detailed report for best model
# =========================
best_variant_fn = variants[best_row['variant']]
X_train_vec, X_test_vec, vocab = make_dataset_for_variant(best_variant_fn)

best_clf = classifiers[best_row['classifier']]
best_clf.fit(X_train_vec, y_train)
best_pred = best_clf.predict(X_test_vec)

print("\n===== Classification Report for Best Model =====")
print(classification_report(y_test, best_pred, digits=4))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!



Variant: BASE
KNN(k=5)        Accuracy = 0.6250
BernoulliNB     Accuracy = 0.8295
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8636
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8068

Variant: STEM
KNN(k=5)        Accuracy = 0.6477
BernoulliNB     Accuracy = 0.8636
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8636
LinearSVC       Accuracy = 0.8409
RandomForest    Accuracy = 0.8068

Variant: LEMMA
KNN(k=5)        Accuracy = 0.6477
BernoulliNB     Accuracy = 0.8523
MultinomialNB   Accuracy = 0.8523
LogReg          Accuracy = 0.8409
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8068

Variant: SPELL
KNN(k=5)        Accuracy = 0.6250
BernoulliNB     Accuracy = 0.8295
MultinomialNB   Accuracy = 0.8295
LogReg          Accuracy = 0.8750
LinearSVC       Accuracy = 0.8182
RandomForest    Accuracy = 0.8182


===== Top Results =====
variant    classifier  accuracy
  SPELL        LogReg  0.875000
   BASE        LogReg  0.863636


###### KEEP STOPWORDS

In [14]:
# =========================================
# Spam / Non-spam Comment Classification
# Correct, leakage-free, spam-aware version
# =========================================

import re
import html
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import nltk

from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# -----------------------------
# NLTK resources
# -----------------------------
for pkg in [
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

# -----------------------------
# 1) Load data
# -----------------------------
df = pd.read_csv("/content/youtube.csv")
df = df[["CONTENT", "CLASS"]].dropna().copy()
df["CONTENT"] = df["CONTENT"].astype(str)
df["CLASS"] = df["CLASS"].astype(int)

X = df["CONTENT"]
y = df["CLASS"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y
)

# -----------------------------
# 2) Spam-aware minimal cleaning
#    - lowercase
#    - url / email / mention normalization
#    - elongation normalization
#    - punctuation markers kept as tokens
# -----------------------------
STOP_WORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

url_re = re.compile(r"(https?://\S+|www\.\S+)")
email_re = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
mention_re = re.compile(r"@\w+")
multi_space_re = re.compile(r"\s+")
elong_re = re.compile(r"(.)\1{2,}")   # coooool -> coool -> cool-ish
non_keep_re = re.compile(r"[^a-zA-Z!? ]+")

def spam_clean(text: str) -> str:
    text = html.unescape(str(text)).lower()
    text = url_re.sub(" url ", text)
    text = email_re.sub(" email ", text)
    text = mention_re.sub(" user ", text)

    # keep useful punctuation as tokens
    text = text.replace("!", " exclam ")
    text = text.replace("?", " quest ")

    # normalize repeated characters
    text = elong_re.sub(r"\1\1", text)

    # remove other symbols
    text = non_keep_re.sub(" ", text)
    text = multi_space_re.sub(" ", text).strip()
    return text

def tokenize(text: str):
    return [t for t in spam_clean(text).split() if t]

# -----------------------------
# 3) POS helper for lemmatization
# -----------------------------
def get_wordnet_pos(treebank_tag: str):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    if treebank_tag.startswith("V"):
        return wordnet.VERB
    if treebank_tag.startswith("N"):
        return wordnet.NOUN
    if treebank_tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens_list):
    try:
        tagged = nltk.pos_tag(tokens_list)
        return [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    except:
        return [lemmatizer.lemmatize(w) for w in tokens_list]

# -----------------------------
# 4) Spell correction built ONLY from train
# -----------------------------
train_tokens_for_spell = []
for txt in X_train_raw:
    train_tokens_for_spell.extend(tokenize(txt))

word_counts = Counter(train_tokens_for_spell)
train_vocab = set(word_counts.keys())
alphabet = "abcdefghijklmnopqrstuvwxyz"

def known(words):
    return set(w for w in words if w in train_vocab)

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [L + R[1:] for L, R in splits if R]
    transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]
    replaces = [L + c + R[1:] for L, R in splits if R for c in alphabet]
    inserts = [L + c + R for L, R in splits for c in alphabet]
    return set(deletes + transposes + replaces + inserts)

def candidates(word):
    return known([word]) or known(edits1(word)) or {word}

def correct_spelling(token):
    # conservative correction:
    # do not touch short words, stopwords, or already-known words
    if len(token) < 4 or token in STOP_WORDS or token in train_vocab or not token.isalpha():
        return token
    cand = candidates(token)
    return max(cand, key=lambda w: word_counts.get(w, 0))

# -----------------------------
# 5) Preprocessing variants
#    Keep stopwords by default.
#    You can compare with/without stopwords.
# -----------------------------
def make_preprocessor(remove_stopwords=False):
    def base(text):
        tokens_list = tokenize(text)
        if remove_stopwords:
            tokens_list = [t for t in tokens_list if t not in STOP_WORDS]
        return " ".join(tokens_list)

    def stem(text):
        tokens_list = tokenize(text)
        if remove_stopwords:
            tokens_list = [t for t in tokens_list if t not in STOP_WORDS]
        tokens_list = [stemmer.stem(t) for t in tokens_list]
        return " ".join(tokens_list)

    def lemma(text):
        tokens_list = tokenize(text)
        if remove_stopwords:
            tokens_list = [t for t in tokens_list if t not in STOP_WORDS]
        tokens_list = lemmatize_tokens(tokens_list)
        return " ".join(tokens_list)

    def spell(text):
        tokens_list = tokenize(text)
        if remove_stopwords:
            tokens_list = [t for t in tokens_list if t not in STOP_WORDS]
        tokens_list = [correct_spelling(t) for t in tokens_list]
        return " ".join(tokens_list)

    return {
        "BASE": base,
        "STEM": stem,
        "LEMMA": lemma,
        "SPELL": spell
    }

# -----------------------------
# 6) Feature extraction
#    Binary word n-grams are good for spam detection.
#    Bigram helps with phrases like "check out", "subscribe now", etc.
# -----------------------------
vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    lowercase=False,
    token_pattern=None,
    binary=True,
    ngram_range=(1, 2),
    min_df=1,
    max_features=2500
)

# -----------------------------
# 7) Classifiers
# -----------------------------
classifiers = {
    "KNN(k=3)": KNeighborsClassifier(n_neighbors=3, metric="cosine", algorithm="brute"),
    "BernoulliNB": BernoulliNB(),
    "MultinomialNB": MultinomialNB(),
    "LogReg": LogisticRegression(max_iter=4000, solver="liblinear"),
    "LinearSVC": LinearSVC(),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=1)
}

# -----------------------------
# 8) Run experiments
#    Compare both stopword modes:
#    - KEEP_STOPWORDS
#    - REMOVE_STOPWORDS
# -----------------------------
all_results = []

for sw_name, remove_sw in [
    ("KEEP_STOPWORDS", False),
    ("REMOVE_STOPWORDS", True),
]:
    preprocessors = make_preprocessor(remove_stopwords=remove_sw)

    for variant_name, preprocess_fn in preprocessors.items():
        print("\n" + "=" * 70)
        print(f"Stopword mode: {sw_name} | Variant: {variant_name}")
        print("=" * 70)

        X_train_proc = [preprocess_fn(t) for t in X_train_raw]
        X_test_proc = [preprocess_fn(t) for t in X_test_raw]

        # Fit vectorizer ONLY on train
        X_train_vec = vectorizer.fit_transform(X_train_proc)
        X_test_vec = vectorizer.transform(X_test_proc)

        for clf_name, clf in classifiers.items():
            if clf_name == "RandomForest":
                clf.fit(X_train_vec.toarray(), y_train)
                pred = clf.predict(X_test_vec.toarray())
            else:
                clf.fit(X_train_vec, y_train)
                pred = clf.predict(X_test_vec)

            acc = accuracy_score(y_test, pred)
            f1 = f1_score(y_test, pred)

            all_results.append({
                "stopwords": sw_name,
                "variant": variant_name,
                "classifier": clf_name,
                "accuracy": acc,
                "f1": f1
            })

            print(f"{clf_name:15s}  Accuracy = {acc:.4f}   F1 = {f1:.4f}")

# -----------------------------
# 9) Best result
# -----------------------------
results_df = pd.DataFrame(all_results).sort_values(
    ["accuracy", "f1"], ascending=False
).reset_index(drop=True)

print("\n\n===== Top Results =====")
print(results_df.head(12).to_string(index=False))

best = results_df.iloc[0]
print("\n===== Best Model =====")
print(f"Stopword mode : {best['stopwords']}")
print(f"Variant       : {best['variant']}")
print(f"Classifier    : {best['classifier']}")
print(f"Accuracy      : {best['accuracy']:.4f}")
print(f"F1            : {best['f1']:.4f}")

# -----------------------------
# 10) Detailed report for the best model
# -----------------------------
best_preprocessor = make_preprocessor(remove_stopwords=(best["stopwords"] == "REMOVE_STOPWORDS"))[best["variant"]]

X_train_proc = [best_preprocessor(t) for t in X_train_raw]
X_test_proc = [best_preprocessor(t) for t in X_test_raw]

X_train_vec = vectorizer.fit_transform(X_train_proc)
X_test_vec = vectorizer.transform(X_test_proc)

best_clf = classifiers[best["classifier"]]

if best["classifier"] == "RandomForest":
    best_clf.fit(X_train_vec.toarray(), y_train)
    best_pred = best_clf.predict(X_test_vec.toarray())
else:
    best_clf.fit(X_train_vec, y_train)
    best_pred = best_clf.predict(X_test_vec)

print("\n===== Classification Report =====")
print(classification_report(y_test, best_pred, digits=4))


Stopword mode: KEEP_STOPWORDS | Variant: BASE
KNN(k=3)         Accuracy = 0.9318   F1 = 0.9318
BernoulliNB      Accuracy = 0.9318   F1 = 0.9286
MultinomialNB    Accuracy = 0.9318   F1 = 0.9318
LogReg           Accuracy = 0.9432   F1 = 0.9398
LinearSVC        Accuracy = 0.9205   F1 = 0.9157
RandomForest     Accuracy = 0.9545   F1 = 0.9524

Stopword mode: KEEP_STOPWORDS | Variant: STEM
KNN(k=3)         Accuracy = 0.9318   F1 = 0.9318
BernoulliNB      Accuracy = 0.9318   F1 = 0.9286
MultinomialNB    Accuracy = 0.9545   F1 = 0.9535
LogReg           Accuracy = 0.9659   F1 = 0.9647
LinearSVC        Accuracy = 0.9205   F1 = 0.9157
RandomForest     Accuracy = 0.9432   F1 = 0.9398

Stopword mode: KEEP_STOPWORDS | Variant: LEMMA
KNN(k=3)         Accuracy = 0.9205   F1 = 0.9213
BernoulliNB      Accuracy = 0.9432   F1 = 0.9412
MultinomialNB    Accuracy = 0.9432   F1 = 0.9425
LogReg           Accuracy = 0.9659   F1 = 0.9647
LinearSVC        Accuracy = 0.9432   F1 = 0.9412
RandomForest     Accuracy